# Notebook 51: Multi-Timeframe Day Trading Research

**Goal:** Find optimal timeframe for active income generation via more frequent trades.

**Hypothesis:** Faster timeframes (4h, 8h, 12h) might generate more trade opportunities while maintaining edge.

**Timeframes tested:**
- 1h (hourly) - most granular
- 4h - common day trading timeframe
- 8h - 3 trades/day max
- 12h - 2 trades/day max
- 1d (daily) - baseline

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
from pathlib import Path

DATA_DIR = Path("../data")
DAILY_DIR = DATA_DIR / "daily"
HOURLY_DIR = DATA_DIR / "hourly"

## 1. Load and Resample Data

In [ ]:
# Load hourly data
def load_hourly():
    price = pd.read_parquet(HOURLY_DIR / "price.parquet")
    sopr = pd.read_parquet(HOURLY_DIR / "sopr.parquet")
    sopr_sth = pd.read_parquet(HOURLY_DIR / "sopr_sth.parquet")
    sopr_lth = pd.read_parquet(HOURLY_DIR / "sopr_lth.parquet")
    realized_loss = pd.read_parquet(HOURLY_DIR / "realized_loss.parquet")
    
    df = price.rename(columns={"value": "price"}).set_index("time")
    df["sopr"] = sopr.set_index("time")["value"]
    df["sopr_sth"] = sopr_sth.set_index("time")["value"]
    df["sopr_lth"] = sopr_lth.set_index("time")["value"]
    df["realized_loss"] = realized_loss.set_index("time")["value"]
    
    return df

df_h1 = load_hourly()
print(f"Hourly data: {len(df_h1):,} rows")
print(f"Date range: {df_h1.index.min()} to {df_h1.index.max()}")

In [ ]:
def resample_to_timeframe(df_h1, timeframe):
    """Resample hourly data to specified timeframe."""
    df = pd.DataFrame()
    
    # Price: close of period
    df["price"] = df_h1["price"].resample(timeframe).last()
    
    # SOPR metrics: mean of period (average behavior)
    df["sopr"] = df_h1["sopr"].resample(timeframe).mean()
    df["sopr_sth"] = df_h1["sopr_sth"].resample(timeframe).mean()
    df["sopr_lth"] = df_h1["sopr_lth"].resample(timeframe).mean()
    
    # Realized loss: sum of period (total capitulation)
    df["realized_loss"] = df_h1["realized_loss"].resample(timeframe).sum()
    
    return df.dropna()

# Create all timeframes
df_4h = resample_to_timeframe(df_h1, "4h")
df_8h = resample_to_timeframe(df_h1, "8h")
df_12h = resample_to_timeframe(df_h1, "12h")
df_1d = resample_to_timeframe(df_h1, "1D")

print(f"1H:  {len(df_h1):,} bars")
print(f"4H:  {len(df_4h):,} bars")
print(f"8H:  {len(df_8h):,} bars")
print(f"12H: {len(df_12h):,} bars")
print(f"1D:  {len(df_1d):,} bars")

In [ ]:
# Calculate z-scores for each timeframe
def add_zscore(df, window_periods):
    """Add realized loss z-score."""
    df = df.copy()
    df["rl_mean"] = df["realized_loss"].rolling(window=window_periods, min_periods=window_periods//2).mean()
    df["rl_std"] = df["realized_loss"].rolling(window=window_periods, min_periods=window_periods//2).std()
    df["rl_zscore"] = (df["realized_loss"] - df["rl_mean"]) / df["rl_std"]
    return df

# 365 days equivalent for each timeframe
df_h1 = add_zscore(df_h1, 365 * 24)      # 8760 hours
df_4h = add_zscore(df_4h, 365 * 6)       # 2190 4H bars
df_8h = add_zscore(df_8h, 365 * 3)       # 1095 8H bars
df_12h = add_zscore(df_12h, 365 * 2)     # 730 12H bars
df_1d = add_zscore(df_1d, 365)           # 365 days

print("Z-scores calculated for all timeframes")

In [ ]:
# Filter to backtest period (2019+)
START_DATE = "2019-01-01"

df_h1 = df_h1[df_h1.index >= START_DATE].dropna()
df_4h = df_4h[df_4h.index >= START_DATE].dropna()
df_8h = df_8h[df_8h.index >= START_DATE].dropna()
df_12h = df_12h[df_12h.index >= START_DATE].dropna()
df_1d = df_1d[df_1d.index >= START_DATE].dropna()

print(f"Backtest period: {START_DATE} to present")
print(f"1H:  {len(df_h1):,} bars")
print(f"4H:  {len(df_4h):,} bars")
print(f"8H:  {len(df_8h):,} bars")
print(f"12H: {len(df_12h):,} bars")
print(f"1D:  {len(df_1d):,} bars")

## 2. Helper Functions

In [ ]:
def get_metrics(pf, years):
    """Extract key metrics from portfolio."""
    trades = pf.trades.records_readable
    if len(trades) == 0:
        return None
    
    total_return = pf.total_return() * 100
    
    # Calculate trades per year
    trades_per_year = len(trades) / years if years > 0 else 0
    
    # Average trade duration
    if "Entry Timestamp" in trades.columns and "Exit Timestamp" in trades.columns:
        durations = (trades["Exit Timestamp"] - trades["Entry Timestamp"]).dropna()
        avg_duration_days = durations.mean().total_seconds() / 86400 if len(durations) > 0 else 0
    else:
        avg_duration_days = 0
    
    return {
        "return": total_return,
        "cagr": ((1 + total_return/100) ** (1/years) - 1) * 100 if years > 0 else 0,
        "sharpe": pf.sharpe_ratio(),
        "max_dd": pf.max_drawdown() * 100,
        "trades": len(trades),
        "trades_per_year": trades_per_year,
        "avg_duration_days": avg_duration_days,
        "win_rate": (trades["PnL"] > 0).mean() * 100,
        "profit_factor": abs(trades[trades["PnL"] > 0]["PnL"].sum() / trades[trades["PnL"] < 0]["PnL"].sum()) if (trades["PnL"] < 0).any() else np.inf,
        "avg_win": trades[trades["PnL"] > 0]["Return"].mean() * 100 if (trades["PnL"] > 0).any() else 0,
        "avg_loss": trades[trades["PnL"] < 0]["Return"].mean() * 100 if (trades["PnL"] < 0).any() else 0,
    }

years = (df_1d.index.max() - df_1d.index.min()).days / 365.25
bh_return = (df_1d["price"].iloc[-1] / df_1d["price"].iloc[0] - 1) * 100
print(f"Backtest: {years:.2f} years | Buy & Hold: {bh_return:+,.0f}%")

## 3. STRAT-002 on All Timeframes

Test the same strategy (SOPR < 1 + STH-SOPR < 1 + RL z-score > 0.5) on each timeframe.

In [ ]:
def run_strat002(df, freq, trail_pct=0.30):
    """Run STRAT-002 on given dataframe."""
    # Entry: SOPR < 1 AND STH-SOPR < 1 AND RL z-score > 0.5
    cond = (df["sopr"] < 1) & (df["sopr_sth"] < 1) & (df["rl_zscore"] > 0.5)
    entry = cond & ~cond.shift(1).fillna(False)
    
    if entry.sum() == 0:
        return None, None
    
    pf = vbt.Portfolio.from_signals(
        close=df["price"],
        entries=entry,
        exits=None,
        sl_stop=trail_pct,
        sl_trail=True,
        freq=freq,
        init_cash=10000,
        fees=0.001
    )
    
    return pf, entry

# Run on all timeframes
results = {}

timeframes = [
    ("1H", df_h1, "1h"),
    ("4H", df_4h, "4h"),
    ("8H", df_8h, "8h"),
    ("12H", df_12h, "12h"),
    ("1D", df_1d, "1d"),
]

for name, df, freq in timeframes:
    pf, entry = run_strat002(df, freq)
    if pf is not None:
        m = get_metrics(pf, years)
        results[name] = {"metrics": m, "pf": pf, "entry": entry, "df": df}
        print(f"{name}: {m['return']:+,.0f}% | {m['trades']} trades | {m['trades_per_year']:.1f}/yr")
    else:
        print(f"{name}: NO ENTRIES")

In [ ]:
# Detailed comparison
print("\n" + "="*140)
print("STRAT-002: MULTI-TIMEFRAME COMPARISON")
print("="*140)
print(f"\n{'TF':<6} {'Return':>12} {'CAGR':>10} {'Sharpe':>8} {'MaxDD':>8} {'Trades':>8} {'Tr/Yr':>8} {'AvgDays':>10} {'WinRate':>10} {'PF':>8}")
print("-"*140)

for name in ["1H", "4H", "8H", "12H", "1D"]:
    if name in results:
        m = results[name]["metrics"]
        print(f"{name:<6} {m['return']:>+11,.0f}% {m['cagr']:>+9.1f}% {m['sharpe']:>8.2f} {m['max_dd']:>7.1f}% {m['trades']:>8} {m['trades_per_year']:>8.1f} {m['avg_duration_days']:>10.1f} {m['win_rate']:>9.0f}% {m['profit_factor']:>8.2f}")

print("-"*140)
print(f"{'B&H':<6} {bh_return:>+11,.0f}%")

## 4. Tighter Trail Stops for Day Trading

30% trail is for swing trading. For day trading, test tighter stops.

In [ ]:
# Test different trail stops on 4H timeframe
trail_stops = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

print("\n" + "="*120)
print("4H TIMEFRAME: TRAIL STOP OPTIMIZATION")
print("="*120)
print(f"\n{'Trail':>8} {'Return':>12} {'CAGR':>10} {'Sharpe':>8} {'MaxDD':>8} {'Trades':>8} {'Tr/Yr':>8} {'AvgDays':>10} {'WinRate':>10}")
print("-"*120)

trail_results_4h = {}

for trail in trail_stops:
    pf, entry = run_strat002(df_4h, "4h", trail_pct=trail)
    if pf is not None:
        m = get_metrics(pf, years)
        trail_results_4h[trail] = {"metrics": m, "pf": pf}
        print(f"{trail*100:>7.0f}% {m['return']:>+11,.0f}% {m['cagr']:>+9.1f}% {m['sharpe']:>8.2f} {m['max_dd']:>7.1f}% {m['trades']:>8} {m['trades_per_year']:>8.1f} {m['avg_duration_days']:>10.1f} {m['win_rate']:>9.0f}%")

In [ ]:
# Test different trail stops on 8H timeframe
print("\n" + "="*120)
print("8H TIMEFRAME: TRAIL STOP OPTIMIZATION")
print("="*120)
print(f"\n{'Trail':>8} {'Return':>12} {'CAGR':>10} {'Sharpe':>8} {'MaxDD':>8} {'Trades':>8} {'Tr/Yr':>8} {'AvgDays':>10} {'WinRate':>10}")
print("-"*120)

trail_results_8h = {}

for trail in trail_stops:
    pf, entry = run_strat002(df_8h, "8h", trail_pct=trail)
    if pf is not None:
        m = get_metrics(pf, years)
        trail_results_8h[trail] = {"metrics": m, "pf": pf}
        print(f"{trail*100:>7.0f}% {m['return']:>+11,.0f}% {m['cagr']:>+9.1f}% {m['sharpe']:>8.2f} {m['max_dd']:>7.1f}% {m['trades']:>8} {m['trades_per_year']:>8.1f} {m['avg_duration_days']:>10.1f} {m['win_rate']:>9.0f}%")

## 5. Income Analysis: Trades per Year vs Return

For income generation, we want:
- More trades per year (more opportunities)
- Consistent positive returns
- Reasonable win rate

In [ ]:
# Income potential analysis
print("\n" + "="*100)
print("INCOME GENERATION ANALYSIS")
print("="*100)
print("\nAssumption: $100,000 capital, compounding returns")
print("\n")

capital = 100000

for name in ["1H", "4H", "8H", "12H", "1D"]:
    if name in results:
        m = results[name]["metrics"]
        
        # Annual return based on CAGR
        annual_return = capital * (m['cagr'] / 100)
        
        # Return per trade
        return_per_trade = m['return'] / m['trades'] if m['trades'] > 0 else 0
        
        print(f"{name}:")
        print(f"  Trades/year:     {m['trades_per_year']:.1f}")
        print(f"  Avg trade:       {return_per_trade:+.1f}%")
        print(f"  Win rate:        {m['win_rate']:.0f}%")
        print(f"  CAGR:            {m['cagr']:+.1f}%")
        print(f"  Annual profit:   ${annual_return:+,.0f}")
        print()

## 6. Trade Distribution Analysis

In [ ]:
# Show trades for each timeframe
for name in ["4H", "8H", "12H", "1D"]:
    if name in results:
        pf = results[name]["pf"]
        trades = pf.trades.records_readable
        
        print(f"\n{'='*80}")
        print(f"{name} TRADES")
        print(f"{'='*80}")
        print(trades[["Entry Timestamp", "Exit Timestamp", "Return", "PnL"]].head(15).to_string())
        print(f"...")
        print(f"Total trades: {len(trades)}")

In [ ]:
# Trade return distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, name in zip(axes.flat, ["4H", "8H", "12H", "1D"]):
    if name in results:
        pf = results[name]["pf"]
        trades = pf.trades.records_readable
        returns = trades["Return"] * 100
        
        ax.hist(returns, bins=30, edgecolor='black', alpha=0.7)
        ax.axvline(x=0, color='red', linestyle='--')
        ax.axvline(x=returns.mean(), color='green', linestyle='-', label=f'Mean: {returns.mean():.1f}%')
        ax.set_title(f"{name}: Trade Returns Distribution")
        ax.set_xlabel("Return (%)")
        ax.set_ylabel("Frequency")
        ax.legend()

plt.tight_layout()
plt.show()

## 7. Relaxed Entry Conditions for More Trades

STRAT-002 is very selective. For day trading, test looser conditions.

In [ ]:
def run_custom_strat(df, freq, sopr_thresh=1.0, sth_thresh=1.0, rl_z_thresh=0.5, trail_pct=0.20):
    """Run strategy with custom thresholds."""
    cond = (df["sopr"] < sopr_thresh) & (df["sopr_sth"] < sth_thresh) & (df["rl_zscore"] > rl_z_thresh)
    entry = cond & ~cond.shift(1).fillna(False)
    
    if entry.sum() == 0:
        return None, 0
    
    pf = vbt.Portfolio.from_signals(
        close=df["price"],
        entries=entry,
        exits=None,
        sl_stop=trail_pct,
        sl_trail=True,
        freq=freq,
        init_cash=10000,
        fees=0.001
    )
    
    return pf, entry.sum()

# Test relaxed conditions on 4H
print("\n" + "="*120)
print("4H: RELAXED ENTRY CONDITIONS FOR MORE TRADES")
print("="*120)

conditions = [
    # (sopr, sth, rl_z, trail, description)
    (1.0, 1.0, 0.5, 0.20, "STRAT-002 base"),
    (1.0, 1.0, 0.0, 0.20, "Remove RL z-score filter"),
    (1.0, 1.0, -0.5, 0.15, "RL z > -0.5 (any stress)"),
    (1.0, 1.05, 0.0, 0.15, "STH < 1.05 (slight loss)"),
    (1.02, 1.0, 0.0, 0.15, "SOPR < 1.02 (near breakeven)"),
]

print(f"\n{'Description':<30} {'Return':>12} {'Trades':>8} {'Tr/Yr':>8} {'WinRate':>10} {'Sharpe':>8}")
print("-"*100)

for sopr, sth, rl_z, trail, desc in conditions:
    pf, n_entries = run_custom_strat(df_4h, "4h", sopr, sth, rl_z, trail)
    if pf is not None:
        m = get_metrics(pf, years)
        print(f"{desc:<30} {m['return']:>+11,.0f}% {m['trades']:>8} {m['trades_per_year']:>8.1f} {m['win_rate']:>9.0f}% {m['sharpe']:>8.2f}")
    else:
        print(f"{desc:<30} {'NO TRADES':>12}")

## 8. SOPR-Only Strategy (Simpler, More Trades)

In [ ]:
def run_sopr_only(df, freq, sopr_thresh=1.0, trail_pct=0.20):
    """Simple SOPR < threshold strategy."""
    cond = df["sopr"] < sopr_thresh
    entry = cond & ~cond.shift(1).fillna(False)
    
    if entry.sum() == 0:
        return None
    
    pf = vbt.Portfolio.from_signals(
        close=df["price"],
        entries=entry,
        exits=None,
        sl_stop=trail_pct,
        sl_trail=True,
        freq=freq,
        init_cash=10000,
        fees=0.001
    )
    
    return pf

print("\n" + "="*100)
print("SOPR-ONLY STRATEGY (Simpler, More Trades)")
print("="*100)

for name, df, freq in [("4H", df_4h, "4h"), ("8H", df_8h, "8h"), ("1D", df_1d, "1d")]:
    print(f"\n{name} - SOPR < 1:")
    for trail in [0.10, 0.15, 0.20, 0.25, 0.30]:
        pf = run_sopr_only(df, freq, 1.0, trail)
        if pf:
            m = get_metrics(pf, years)
            print(f"  Trail {trail*100:.0f}%: {m['return']:+,.0f}% | {m['trades']} trades | {m['trades_per_year']:.1f}/yr | {m['win_rate']:.0f}% win")

## 9. Final Summary

In [ ]:
print("\n" + "="*100)
print("FINAL SUMMARY: DAY TRADING VIABILITY")
print("="*100)

print("""
KEY FINDINGS:

1. TRADE FREQUENCY:
   - Daily STRAT-002 generates ~1-2 trades/year (swing trading)
   - Faster timeframes generate MORE signals but MORE noise
   
2. OPTIMAL TIMEFRAME FOR DAY TRADING:
""")

# Find best risk-adjusted return
best_tf = None
best_sharpe = -999
for name in results:
    if results[name]["metrics"]["sharpe"] > best_sharpe:
        best_sharpe = results[name]["metrics"]["sharpe"]
        best_tf = name

print(f"   Best risk-adjusted (Sharpe): {best_tf}")

# Find most trades
most_trades = max(results.keys(), key=lambda x: results[x]["metrics"]["trades"])
print(f"   Most trades: {most_trades} ({results[most_trades]['metrics']['trades']} trades)")

# Find best return
best_return = max(results.keys(), key=lambda x: results[x]["metrics"]["return"])
print(f"   Best return: {best_return} ({results[best_return]['metrics']['return']:+,.0f}%)")

print("""
3. RECOMMENDATION:
""")

if results.get("1D", {}).get("metrics", {}).get("return", 0) > results.get("4H", {}).get("metrics", {}).get("return", 0):
    print("   → STICK WITH DAILY for best returns")
    print("   → On-chain signals work better with daily aggregation")
    print("   → Day trading with on-chain data has more noise than signal")
else:
    print("   → Consider 4H/8H for more active trading")
    print("   → Use tighter stops for faster exits")

In [ ]:
# Equity curves comparison
fig, ax = plt.subplots(figsize=(14, 6))

for name in ["4H", "8H", "12H", "1D"]:
    if name in results:
        pf = results[name]["pf"]
        m = results[name]["metrics"]
        pf.value().resample('D').last().plot(ax=ax, label=f"{name} ({m['return']:+,.0f}%, {m['trades']} trades)")

ax.set_title("STRAT-002: Multi-Timeframe Comparison")
ax.set_ylabel("Portfolio Value ($)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()